# Exemplo 04 - Mutacao Aplicada: Descobrindo uma Senha

---

## O Problema

Imagine que voce precisa **descobrir uma palavra secreta**. Voce nao sabe qual e, mas pode verificar o quao **proxima** cada tentativa esta da resposta correta.

## Onde isso aparece na vida real?

| Area | Exemplo |
|------|--------|
| Otimizacao | Encontrar a melhor configuracao de parametros |
| Seguranca | Simulacao de ataques a senhas |
| Design | Gerar textos, musicas, layouts |
| Engenharia | Calibrar modelos ate encontrar o melhor ajuste |

## Representacao Genetica

- Cada individuo (cromossomo) e uma **lista de caracteres** (letras)
- Exemplo: `['P', 'Y', 'T', 'H', 'O', 'N']`
- Fitness = quantas letras estao na **posicao correta**

## O que e a mutacao?

A mutacao percorre cada **gene** (letra) e, com uma certa probabilidade, **troca** essa letra por outra aleatoria.

```
Antes:    ['A', 'L', 'X', 'O', 'R', 'I', 'T', 'Z', 'O']
Mutacao no gene 2 e 7:
Depois:   ['A', 'L', 'G', 'O', 'R', 'I', 'T', 'M', 'O']
                       ^                          ^
                  X -> G                     Z -> M
```

In [1]:
import random
import string

---
## Passo 1: Definir o problema

In [13]:
# A palavra secreta que queremos descobrir (o "alvo")
PALAVRA_SECRETA = "EDWILLIE"

# Caracteres possiveis para cada gene (letras maiusculas + espaco)
CARACTERES = string.ascii_uppercase + " "

# Tamanho do cromossomo = tamanho da palavra
TAM_CROMOSSOMO = len(PALAVRA_SECRETA)

print("Palavra secreta:", "?" * TAM_CROMOSSOMO, "(", TAM_CROMOSSOMO, "caracteres )")
print("Caracteres possiveis:", len(CARACTERES))
print("Chance de acertar ao acaso:", round(1 / len(CARACTERES) ** TAM_CROMOSSOMO, 20))

Palavra secreta: ???????? ( 8 caracteres )
Caracteres possiveis: 27
Chance de acertar ao acaso: 3.54070616e-12


A chance de acertar por sorte e **praticamente zero**! Por isso precisamos de um AG.

---
## Passo 2: Funcoes basicas

In [14]:
def criar_individuo():
    """Cria um cromossomo aleatorio (lista de caracteres)."""
    return [random.choice(CARACTERES) for _ in range(TAM_CROMOSSOMO)]


def calcular_fitness(individuo):
    """
    Conta quantas letras estao na posicao correta.
    Fitness maximo = tamanho da palavra.
    """
    acertos = 0
    for i in range(TAM_CROMOSSOMO):
        if individuo[i] == PALAVRA_SECRETA[i]:
            acertos += 1
    return acertos


def mostrar_individuo(individuo):
    """Mostra o cromossomo como uma palavra legivel."""
    return "".join(individuo)


# Teste: criar um individuo e avaliar
teste = criar_individuo()
print("Individuo aleatorio:", mostrar_individuo(teste))
print("Fitness:", calcular_fitness(teste), "/", TAM_CROMOSSOMO)

Individuo aleatorio: TNVYAH S
Fitness: 0 / 8


---
## Passo 3: Funcao de Mutacao

Para cada gene (letra), ha uma probabilidade de ele ser **trocado** por outro caractere aleatorio.

In [4]:
def mutacao(individuo, taxa_mutacao):
    """
    Aplica mutacao em cada gene do cromossomo.

    Parametros:
        individuo (list): Cromossomo (lista de caracteres)
        taxa_mutacao (float): Probabilidade de mutacao por gene (0 a 1)

    Retorna:
        list: Cromossomo possivelmente modificado
    """
    for i in range(TAM_CROMOSSOMO):
        if random.random() < taxa_mutacao:
            # Troca este gene por um caractere aleatorio
            individuo[i] = random.choice(CARACTERES)
    return individuo

print("Funcao mutacao() definida!")

Funcao mutacao() definida!


---
## Passo 4: Demonstracao da mutacao

Vamos aplicar mutacao num mesmo individuo com **taxas diferentes** e ver o efeito.

**Rode varias vezes** para ver resultados diferentes!

In [15]:
original = criar_individuo()
print("Individuo original: ", mostrar_individuo(original))
print("Fitness:            ", calcular_fitness(original), "/", TAM_CROMOSSOMO)
print()

# Taxa baixa: poucas mudancas
print("--- Taxa de mutacao: 10% (baixa) ---")
for i in range(5):
    copia = original[:]
    mutado = mutacao(copia, 0.10)
    print("  Tentativa", i+1, ":", mostrar_individuo(mutado), "  fitness:", calcular_fitness(mutado))

print()

# Taxa alta: muitas mudancas
print("--- Taxa de mutacao: 50% (alta) ---")
for i in range(5):
    copia = original[:]
    mutado = mutacao(copia, 0.50)
    print("  Tentativa", i+1, ":", mostrar_individuo(mutado), "  fitness:", calcular_fitness(mutado))

Individuo original:  GGGGNRMF
Fitness:             0 / 8

--- Taxa de mutacao: 10% (baixa) ---
  Tentativa 1 : NGGGNRMF   fitness: 0
  Tentativa 2 : GGGGNRMF   fitness: 0
  Tentativa 3 : GGGGNRMF   fitness: 0
  Tentativa 4 : GGGGNRMF   fitness: 0
  Tentativa 5 : GGGGNRMF   fitness: 0

--- Taxa de mutacao: 50% (alta) ---
  Tentativa 1 : GGGGN AG   fitness: 0
  Tentativa 2 : GGTMNFDF   fitness: 0
  Tentativa 3 : GGGINRZT   fitness: 1
  Tentativa 4 : JGTGNPCX   fitness: 0
  Tentativa 5 : MJGIPRXB   fitness: 1


**Observe:**
- Com **10%**, poucas letras mudam (o individuo se parece com o original)
- Com **50%**, quase metade das letras muda (o individuo fica muito diferente)

---
## Passo 5: Evolucao com mutacao + selecao

Agora vamos juntar **mutacao + selecao** para ver a palavra **evoluir** letra por letra ate encontrar a senha!

Este e um AG simplificado que demonstra o poder da mutacao.

In [6]:
def evoluir(taxa_mutacao, tamanho_pop=50, max_geracoes=500):
    """
    Evolui uma populacao para descobrir a palavra secreta.
    Retorna o numero de geracoes necessarias (-1 se nao encontrou).
    """
    # Criar populacao inicial aleatoria
    populacao = [criar_individuo() for _ in range(tamanho_pop)]

    for geracao in range(max_geracoes):
        # Calcular fitness de todos
        fitness_lista = [calcular_fitness(ind) for ind in populacao]

        # Encontrar o melhor da geracao
        melhor_fitness = max(fitness_lista)
        melhor_indice = fitness_lista.index(melhor_fitness)
        melhor = populacao[melhor_indice]

        # Mostrar progresso
        if geracao % 50 == 0 or melhor_fitness == TAM_CROMOSSOMO:
            print("  Geracao", geracao, ":", mostrar_individuo(melhor), "  fitness:", melhor_fitness, "/", TAM_CROMOSSOMO)

        # Encontrou a palavra!
        if melhor_fitness == TAM_CROMOSSOMO:
            return geracao

        # Selecao + mutacao para criar nova geracao
        nova_populacao = [melhor[:]]  # Elitismo

        while len(nova_populacao) < tamanho_pop:
            # Selecao por torneio (escolhe 3, pega o melhor)
            candidatos = random.sample(range(tamanho_pop), 3)
            melhor_cand = max(candidatos, key=lambda x: fitness_lista[x])
            filho = populacao[melhor_cand][:]

            # Aplicar mutacao
            filho = mutacao(filho, taxa_mutacao)
            nova_populacao.append(filho)

        populacao = nova_populacao

    return -1

print("Funcao evoluir() definida!")

Funcao evoluir() definida!


---
## Passo 6: Comparar taxas de mutacao

Vamos testar **4 cenarios** para ver o impacto da taxa de mutacao:

In [16]:
# Teste 1: Taxa baixa (3%)
print("--- Taxa de mutacao: 3% (baixa) ---")
g = evoluir(taxa_mutacao=0.03)
if g >= 0:
    print("  Encontrou em", g, "geracoes!")
else:
    print("  Nao encontrou em 500 geracoes")

--- Taxa de mutacao: 3% (baixa) ---
  Geracao 0 : WWJIWLHD   fitness: 2 / 8
  Geracao 50 : LDWILLIE   fitness: 7 / 8
  Geracao 65 : EDWILLIE   fitness: 8 / 8
  Encontrou em 65 geracoes!


In [17]:
# Teste 2: Taxa equilibrada (10%)
print("--- Taxa de mutacao: 10% (equilibrada) ---")
g = evoluir(taxa_mutacao=0.10)
if g >= 0:
    print("  Encontrou em", g, "geracoes!")
else:
    print("  Nao encontrou em 500 geracoes")

--- Taxa de mutacao: 10% (equilibrada) ---
  Geracao 0 : ZVA LMGE   fitness: 2 / 8
  Geracao 50 : EDQILLIE   fitness: 7 / 8
  Geracao 51 : EDWILLIE   fitness: 8 / 8
  Encontrou em 51 geracoes!


In [18]:
# Teste 3: Taxa alta (50%)
print("--- Taxa de mutacao: 50% (alta - muito caos!) ---")
g = evoluir(taxa_mutacao=0.50)
if g >= 0:
    print("  Encontrou em", g, "geracoes!")
else:
    print("  Nao encontrou em 500 geracoes (taxa muito alta = caos)")

--- Taxa de mutacao: 50% (alta - muito caos!) ---
  Geracao 0 : YDYWLXRZ   fitness: 2 / 8
  Geracao 50 : UVNILJIE   fitness: 4 / 8
  Geracao 100 : AHWILXIE   fitness: 5 / 8
  Geracao 150 : AHWILXIE   fitness: 5 / 8
  Geracao 200 : EHWILXIE   fitness: 6 / 8
  Geracao 250 : EHWILXIE   fitness: 6 / 8
  Geracao 300 : EHWILXIE   fitness: 6 / 8
  Geracao 350 : EHWILXIE   fitness: 6 / 8
  Geracao 400 : EHWILXIE   fitness: 6 / 8
  Geracao 450 : EHWILXIE   fitness: 6 / 8
  Nao encontrou em 500 geracoes (taxa muito alta = caos)


---
## Passo 7: O que acontece SEM mutacao?

Este e o teste mais importante! Sem mutacao (taxa = 0%), o AG **nao consegue** introduzir novos genes. A populacao converge para uma solucao e **para de evoluir**.

In [19]:
print("--- SEM MUTACAO (taxa = 0%) ---")
print()
g = evoluir(taxa_mutacao=0.0, max_geracoes=100)
if g >= 0:
    print("  Encontrou em", g, "geracoes!")
else:
    print("  NAO encontrou! O AG ficou PRESO.")
    print("  Sem mutacao, nao ha como introduzir genes novos.")
    print("  A populacao converge e para de evoluir.")

--- SEM MUTACAO (taxa = 0%) ---

  Geracao 0 : EFVPOIIC   fitness: 2 / 8
  Geracao 50 : EFVPOIIC   fitness: 2 / 8
  NAO encontrou! O AG ficou PRESO.
  Sem mutacao, nao ha como introduzir genes novos.
  A populacao converge e para de evoluir.


---
## Conclusao

| Taxa de mutacao | Comportamento |
|:---:|:---|
| **0% (nenhuma)** | AG fica **PRESO** - nao consegue evoluir |
| **3% (baixa)** | Evolui devagar, mas e estavel |
| **10% (equilibrada)** | Encontra **rapido**! Melhor balanco |
| **50% (alta)** | Muito caos, destroi boas solucoes, dificil convergir |

### A mutacao e essencial porque:

1. **Introduz diversidade** - cria genes que nao existiam na populacao
2. **Evita convergencia prematura** - impede que todos fiquem iguais
3. **Permite escapar de otimos locais** - explora novas regioes do espaco de busca


### Exercicio:

Mude a `PALAVRA_SECRETA` para o seu nome e rode novamente!

In [ ]:
# EXERCICIO: Mude a palavra secreta e rode!
# PALAVRA_SECRETA = "SEU NOME"
# TAM_CROMOSSOMO = len(PALAVRA_SECRETA)
# g = evoluir(taxa_mutacao=0.10)
# print("Encontrou em", g, "geracoes!")